
# Construção do `gerador.cpp` e `sorts.cpp`

Este notebook descreve a construção dos dois principais programas do projeto:

1. `gerador.cpp`
2. `sorts.cpp`

O objetivo é documentar:
- estrutura dos programas;
- metodologia experimental;
- geração dos datasets;
- execução dos benchmarks.

Foi utilizado o agente de IA do ChatGPT para ajudar a fazer ambos programas.



# Objetivo Geral do Projeto

O projeto busca comparar algoritmos de ordenação utilizando diferentes padrões de acesso à memória e diferentes estruturas de entrada.

Os algoritmos avaliados são:

- HeapSort
- MergeSort
- QuickSort



# Construção do `gerador.cpp`

O `gerador.cpp` é responsável por:

1. Criar os datasets;
2. Garantir reprodutibilidade;
3. Salvar os vetores em formato binário.

O programa permite seeds de entrada para possibilitar a reprodutibilidade. Dando default para seed 1 caso nenhum seed de entrada seja informada.



# Estrutura dos Datasets

O gerador cria 8 vetores de tamanho definido pela entrada de usuário:

- ordered
- reverse
- random
- repeated
- repeated_1_5
- zigzag
- noise_end
- random_swaps

Cada dataset foi projetado para provocar diferentes comportamentos dos algoritmos.



# Formato Binário

Os vetores são armazenados em formato `.bin`.

Isso foi escolhido porque:
- leitura é mais rápida;
- arquivos ocupam menos espaço;
- reduz overhead experimental.



# Construção do `sorts.cpp`

O `sorts.cpp` realiza:

1. Leitura dos datasets;
2. Aplicação ou não de estress;
3. Execução dos algoritmos;
4. Medição de métricas;
5. Salvamento em CSV.



# Entrada por Linha de Comando

O programa é executado no formato:

```bash
./sorts <algoritmo> <arquivo_bin> stress carga
```

Exemplo:

```bash
./sorts heap datasets/random.bin cpu 0.5
```



# Isolamento Experimental

Cada execução do benchmark realiza apenas:

- um algoritmo de ordenação;
- um conjunto de dados (dataset);
- uma configuração específica de estresse (none, cpu, ram ou both);
- um único nível de carga (0,5 ou 1,0, quando aplicável).

Os mecanismos de estresse são executados em threads separadas do algoritmo de ordenação. Para estresse de RAM, é criada uma única thread responsável por alocar e acessar continuamente uma quantidade de memória proporcional ao nível de carga configurado. Para estresse de CPU, é criado um conjunto de threads proporcional à carga especificada e ao número de processadores lógicos disponíveis, mantendo a utilização do processador durante toda a execução do benchmark. No cenário "both", os dois mecanismos são executados simultaneamente.

Essa estratégia garante que cada benchmark avalie apenas uma combinação específica de algoritmo, dataset, configuração de estresse e nível de carga, proporcionando:

- isolamento entre os experimentos;
- medições mais confiáveis de tempo, memória e consumo de energia;
- redução da interferência entre diferentes execuções;
- maior reprodutibilidade dos resultados.



# Medição de Tempo

A medição utiliza:

```cpp
std::chrono::high_resolution_clock
```

Os tempos são armazenados em milissegundos.

# Uso de Memória RAM

A medição utiliza
```cpp
VmHWM (pico de uso RAM)
```
Também é medida RAM média

# Gasto Energético

A medição utiliza
```cpp
intel-rapl:0/energy_uj
```

Desativada temporariamente para testagem

# Comparações

A métrica de comparações contabiliza o número total de vezes em que duas entradas são avaliadas em operações condicionais do algoritmo (ex: a < b, a > b, a <= b).

Cada avaliação lógica relevante é instrumentada para incrementar um contador global de comparações.

Essa métrica permite avaliar a complexidade real do algoritmo independente do hardware.

# Swaps

A métrica de swaps contabiliza o número total de trocas de posições realizadas entre elementos do conjunto de dados.

Cada operação de troca (swap(a, b)) incrementa um contador global.

Essa métrica permite medir o custo de reorganização dos dados durante a execução do algoritmo, sendo especialmente relevante para algoritmos de ordenação como QuickSort e HeapSort.

# Estresse de CPU

Para avaliar o impacto da contenção pelos recursos de processamento, foi implementado um mecanismo de geração de carga sobre a CPU executado em paralelo ao algoritmo de ordenação.

Cada thread de estresse realiza continuamente operações aritméticas de ponto flutuante (multiplicação, divisão, soma e subtração) sobre uma variável declarada como volatile, impedindo que o compilador elimine essas operações durante as otimizações. Dessa forma, as unidades de execução da CPU permanecem ocupadas durante toda a execução do benchmark.

O número de threads de estresse não é fixo. Ele é determinado dinamicamente em função do nível de carga especificado e do número de processadores lógicos disponíveis na máquina. Para isso, o programa utiliza std::thread::hardware_concurrency(), calculando a quantidade de threads proporcional ao nível de carga selecionado. Assim, cargas maiores resultam em maior utilização da capacidade de processamento disponível.

Antes do início da medição do benchmark, o programa aguarda todas as threads de CPU iniciarem sua execução, garantindo que o algoritmo de ordenação seja executado sob o nível de estresse desejado desde o primeiro instante.

void runCpuStress(std::atomic<bool>& running,
                  std::atomic<int>& readyCounter)
{
    volatile double x = 1.1;

    readyCounter++;

    while (running)
    {
        x *= 1.0000001;
        x /= 1.00000009;
        x += 0.1234;
        x -= 0.5678;
        x *= 1.0000003;
        x /= 1.0000002;
        x += x * 0.000001;
    }
}

Essa abordagem permite controlar a intensidade da contenção do processador de forma escalável, reproduzindo diferentes níveis de utilização da CPU durante os experimentos.

# Estresse de Memória RAM

Para avaliar o impacto da contenção pelo subsistema de memória, foi implementada uma thread dedicada à geração contínua de acessos à memória principal.

A quantidade de memória utilizada é proporcional ao nível de carga definido para o experimento. O tamanho do buffer é calculado como uma fração de 32 GB, permitindo representar diferentes intensidades de utilização da memória. O buffer é completamente alocado e inicializado antes do início da medição do benchmark.

Durante a execução, os endereços acessados são gerados por meio do algoritmo XorShift64, produzindo uma sequência pseudoaleatória que reduz a localidade espacial dos acessos e dificulta a atuação dos mecanismos de pré-busca (hardware prefetching).

Cada iteração realiza sucessivas operações de leitura e escrita sobre posições pseudoaleatórias do buffer, mantendo elevada a utilização da largura de banda da memória durante toda a execução do algoritmo de ordenação.

Antes de iniciar a medição, o programa aguarda a conclusão da alocação e inicialização do buffer, assegurando que o tempo medido corresponda exclusivamente à execução do algoritmo sob as condições de estresse previamente estabelecidas.

void runRamStress(std::atomic<bool>& running,
                  const float carga,
                  std::atomic<bool>& ready)
{
    const size_t bytesAlvo =
        static_cast<size_t>(32ULL * 1024ULL * 1024ULL * 1024ULL * carga);

    std::vector<unsigned char> buffer(bytesAlvo, 0);

    ready = true;

    uint64_t state = 0x123456789ABCDEFULL;
    unsigned char value = 0;

    while (running)
    {
        for (int i = 0; i < 8; i++)
        {
            state ^= state << 13;
            state ^= state >> 7;
            state ^= state << 17;

            size_t index = state % bytesAlvo;

            value ^= buffer[index];
            buffer[index] = value;
        }
    }
}

Essa implementação produz um fluxo contínuo de acessos pseudoaleatórios à memória principal, aumentando a pressão sobre a largura de banda da RAM e permitindo analisar como algoritmos com diferentes padrões de acesso à memória respondem a diferentes níveis de contenção desse recurso. Além disso, como a alocação do buffer ocorre antes do início da cronometragem, o tempo de benchmark reflete apenas a execução do algoritmo de ordenação sob o estresse configurado.

# Validação Sorting

Todo sorting antes de ser salvo é validado que foi realizado corretamente, sortings errados não são salvos nas estatísticas


# Resultados CSV

Os resultados são salvos em:

```text
results/(tamanho vetor).csv
```

Formato:

```csv
vector_type,algorithm,time_ms,stress,ram_mb,energy_j
```

Também são criados .csv únicos a tipos de vetor e tipos de algoritmo
